# Otimização Optuna

In [ ]:
%load_ext autoreload
%autoreload 2

from src.Classification.Optimizer import ClassificationOptunaOptimizer
from src.Data.Processor import DataStreamProcessor
import pandas as pd

# Definição dos datasets
categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']
datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

features = [
    'Fwd Packet Length Min',
    'Total Fwd Packets',
    'Fwd Packet Length Max',
    'Packet Length Variance',
    'Init_Win_bytes_forward',
    'Flow IAT Mean',
    'Fwd Packets/s',
    'Fwd Packet Length Std',
    'Flow Duration',
    'Total Backward Packets',
    'URG Flag Count',
    'Init_Win_bytes_backward',
    'Flow IAT Min',
    'Bwd Packets/s',
    'Bwd IAT Mean',
    'Down/Up Ratio',
    'Bwd IAT Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Max',
    'Fwd Header Length',
    'Total Length of Fwd Packets',
    'ACK Flag Count',
    'Active Mean',
    'Fwd Packet Length Mean',
    'Fwd PSH Flags',
]

for dataset_path in datasets:
    print(f"\nIniciando otimização para: {dataset_path}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="StandardScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    optimizer = ClassificationOptunaOptimizer(
        stream=stream,
        n_trials=5,
        target_class=0,
        n_runs=5
    )

    melhor_lb = optimizer.optimize('LB')
    # melhot_hat = optimizer.optimize('HAT')
    # melhot_arf = optimizer.optimize('ARF')
    # melhot_ht = optimizer.optimize('HT')


# Execução Pipeline

In [ ]:
%load_ext autoreload
%autoreload 2

from src.Classification.Pipeline import ClassificationExperimentRunner
from src.Classification.Models import get_classification_models
from src.Data.Processor import DataStreamProcessor
import pandas as pd

categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']
datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

features = [
    'Fwd Packet Length Min',
    'Total Fwd Packets',
    'Fwd Packet Length Max',
    'Packet Length Variance',
    'Init_Win_bytes_forward',
    'Flow IAT Mean',
    'Fwd Packets/s',
    'Fwd Packet Length Std',
    'Flow Duration',
    'Total Backward Packets',
    'URG Flag Count',
    'Init_Win_bytes_backward',
    'Flow IAT Min',
    'Bwd Packets/s',
    'Bwd IAT Mean',
    'Down/Up Ratio',
    'Bwd IAT Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Max',
    'Fwd Header Length',
    'Total Length of Fwd Packets',
    'ACK Flag Count',
    'Active Mean',
    'Fwd Packet Length Mean',
    'Fwd PSH Flags',
]

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False, selected_features=features)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="StandardScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_classification_models(
        stream.get_schema(),
        selected_models=['LB']
    )
    
    runner = ClassificationExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_classification_evaluation(
        stream,
        algorithms=algoritmos,
        window_size=1050,
        warmup_instances=0,
        title=nome_experimento,
        target_class=0
    )